# Baseline — najprostsze podejście (wersja 0)

Najnaiwniejszy sposób: **wszystkie akta w jeden prompt** + polecenie „jesteś prawnikiem, napisz apelację". Bez planu, bez podziału na kroki, bez selekcji dokumentów.

Ten notebook: generuje apelację baseline i **ewaluuje** ją (pokrycie + halucynacje), żeby było co porównać z agentem liniowym (`linear_walkthrough.ipynb`) i planerem (`planner_walkthrough.ipynb`).

> Odpowiednik `baseline/main.py`, ale interaktywnie.

## 0. Konfiguracja

Domyślnie lokalnie przez Ollamę (`qwen2.5:14b`). Dla OpenAI — odkomentuj sekcję A, zakomentuj B.

In [ ]:
import os
import sys

# Notebook leży w notebooks/ — przejdź do katalogu głównego projektu, żeby
# działały importy `src.*`/`baseline.*` oraz względne ścieżki do data/.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

# ── B. Lokalnie przez Ollamę (domyślnie) ──
os.environ["LLM_BASE_URL"] = "http://localhost:11434/v1"
os.environ["LLM_API_KEY"]  = "ollama"
os.environ["LLM_MODEL"]    = "qwen2.5:14b"

# ── A. Własny klucz API (OpenAI) ──
# os.environ["LLM_BASE_URL"] = "https://api.openai.com/v1"
# os.environ["LLM_API_KEY"]  = "sk-...wpisz-swój-klucz..."
# os.environ["LLM_MODEL"]    = "gpt-4o-mini"

print("katalog roboczy:", os.getcwd())
print("model:", os.environ["LLM_MODEL"])

## 1. Wczytanie akt i rozmiar promptu

Baseline wrzuca **całość** w jeden prompt. Policzmy, ile to tokenów — to sedno problemu tego podejścia.

In [ ]:
from src.loader import load_all
from src.tokens import count_tokens
from baseline.main import build_context
from baseline.prompts import SYSTEM_PROMPT

docs = load_all()
context = build_context(docs)
prompt_tokens = count_tokens(SYSTEM_PROMPT + "\n\n" + context)

print(f"Dokumentów: {len(docs)}")
print(f"Znaków w kontekście: {len(context):,}")
print(f"Tokenów w promptcie (wejście): ~{prompt_tokens:,}")

## 2. Generowanie apelacji (jeden prompt)

Jedno wywołanie LLM — całe akta wchodzą, wychodzi gotowa apelacja.

In [ ]:
from baseline.main import generate_appeal

appeal = generate_appeal(docs).tekst
print(f"Apelacja: {len(appeal):,} znaków\n")
print(appeal[:1000], "...")

## 3. Pokrycie — czy porusza wymagane zagadnienia?

Sędzia-LLM ocenia względem `data/eval.json`.

In [ ]:
from src.eval.coverage import evaluate

cov = evaluate(appeal)
print(f"Pokrycie: {cov.covered}/{cov.total} = {cov.score:.0%}\n")
for r in cov.results:
    print(("\u2705" if r.covered else "\u274c"), r.issue[:80])

## 4. Halucynacje — czy nie zmyśla faktów? (opcjonalnie, wolne)

Sprawdza, czy fakty w apelacji mają oparcie w aktach (etapowo: ekstrakcja → wybór plików po opisach → weryfikacja).

In [ ]:
from src.eval.grounding import evaluate_grounding

g = evaluate_grounding(appeal, docs)
print(f"Twierdzeń: {g.total}  |  potwierdzone: {g.supported}  |  bez oparcia: {g.unsupported}  |  sprzeczne: {g.contradicted}")
print(f"Wskaźnik halucynacji: {g.hallucination_rate:.0%}\n")
for r in g.results:
    if r.status != "supported":
        print(f"[{r.status}] {r.claim}")

## Podsumowanie

Baseline = **1 wywołanie LLM** z ogromnym promptem (patrz tokeny w sekcji 1). Szybki do napisania, ale: zapchany kontekst, „lost in the middle", brak śladu rozumowania i podatność na pominięcia/halucynacje.

Porównaj wynik z `linear_walkthrough.ipynb` (agent liniowy) i `planner_walkthrough.ipynb`. Tabela wszystkich podejść obok siebie: `python -m src.eval.compare`.